# Revised morphology and cross-lingual abusive-language pipeline

This notebook is a thin Colab launcher for the version-controlled scripts. It replaces the stateful legacy notebook. Smoke-test outputs are never paper results, and the full matrix must run on a GPU runtime.


## 1. Runtime and repository

In Colab select **Runtime → Change runtime type → A100 GPU**. Set `REPO_ROOT` to the repository folder after cloning/uploading it. If the repository is stored in Drive, mount Drive and point to that directory.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# EDIT THIS PATH if needed.
REPO_ROOT = Path("/content/drive/MyDrive/HBB_multilingual")
assert REPO_ROOT.exists(), f"Repository not found: {REPO_ROOT}"
%cd {REPO_ROOT}


## 2. Install exact pipeline dependencies


In [ ]:
!python -m pip install -q -r requirements-analysis.txt -r requirements-experiments.txt -r requirements-morphology.txt


## 3. Rebuild the verified CPU analyses

This audits the local workbooks, fetches pinned Korean/English data, prepares fixed manifests, computes tokenization statistics and linear controls, generates bootstrap intervals and figures, and runs tests.


In [ ]:
!PYTHON=python scripts/run_verified_analysis.sh


## 4. Inspect corrected data and verified results


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

display(Markdown(Path("artifacts/datasets/PAPER_DATA_TABLES.md").read_text()))
display(Markdown(Path("artifacts/tokenization/REPORT.md").read_text()))
display(Markdown(Path("artifacts/linear_diagnostics/REPORT.md").read_text()))


## 5. Add Hungarian before the five-language run

The final five-language experiment must not proceed until the exact Hungarian source file is available. Place it in Drive and rerun preparation with the actual text/label columns. The example below intentionally requires you to edit the path and columns.


In [ ]:
# Example only — edit before running.
# HU_FILE = Path("/content/drive/MyDrive/SIGTURK/hungarian_toxic.csv")
# !python scripts/prepare_standardized_datasets.py \
#   --korean-repo data/raw/korean-hate-speech \
#   --tweeteval-repo data/raw/tweeteval \
#   --hungarian "{HU_FILE}" \
#   --hungarian-text-column text \
#   --hungarian-label-column toxic \
#   --hungarian-threshold 2.5


## 6. Build lemma and UD-morphology views

This downloads Stanza models and creates surface-aligned lemma, morphology, and randomized-morphology placebo views. Manually audit at least 100 analyses per language before interpreting model differences.


In [ ]:
!python scripts/build_ud_morph_views.py \
  data/processed/standardized_all.csv \
  data/processed/standardized_with_morphology.parquet \
  --download --use-gpu --batch-documents 32


## 7. Inspect the 30-run confirmatory plan


In [ ]:
!python scripts/run_experiment_matrix.py
plan = pd.read_json("artifacts/experiment_plan.json")
display(plan[["index", "block", "run_id"]])


## 8. Execute GPU blocks

Before the final paper run, replace every `model_revision: main` in `configs/experiment_matrix.json` with the model's immutable Hugging Face commit hash. Runs resume by skipping completed `metrics.csv` files.


In [ ]:
# Uncomment only after checking immutable model revisions and GPU availability.
# !python scripts/run_experiment_matrix.py --execute


## 9. Aggregate only complete three-seed results

The aggregator fails when any condition is missing seed 42, 123, or 456 and excludes smoke tests.


In [ ]:
# Run after every planned job completes.
# !python scripts/aggregate_encoder_results.py
# display(Markdown(Path("artifacts/results/ENCODER_RESULTS.md").read_text()))


## 10. Blinded linguistic error sample

Choose one completed model and build balanced TP/TN/FP/FN samples. Give annotators only `annotation_blind.csv`.


In [ ]:
# Example after a real run:
# PREDICTIONS = "outputs/encoder/<RUN_ID>/predictions.csv"
# !python scripts/sample_linguistic_errors.py "$PREDICTIONS" --per-cell 25 --seed 42
